**Sample ID**: 151

**Query**:

Hi, I need to make a few exchanges. I want to exchange a bicycle for a larger frame size and a jigsaw puzzle for one with 1000 more pieces, preferably with an art theme. I also need to exchange a camera for one with a lower resolution and cancel my skateboard order.

**DB Type**: Base Case

**Case Description**:

The user, Sofia Li (zip code 78260), is requesting to exchange items from two separate orders. For order #W3916020, she wants to exchange a Bicycle (item '7758198585') for a model with a larger frame size ('5606522780') and a Jigsaw Puzzle (item '4068787148') for one with 1000 more pieces and an art theme ('5546244844'). For order #W4689314, she wants to exchange a Camera (item '5996159312') for a model with a slightly lower resolution ('8363011723'). For all transactions, any price difference or refund should be processed to her Visa card ('credit_card_3951670'). She also wants to cancel a skateboard order, but has stated that if a single item cannot be canceled, she will cancel the entire order herself on the website.





```
<multiturn info>
User Info: Sofia Li, zip 78260 (Information Gathering)
Item 1 Exchange: Bicycle for a larger frame size for my kid (Information Gathering)
Item 2 Exchange: Jigsaw Puzzle for one with the same difficulty but 1000 more pieces (Information Gathering)
Item 2 Preference: Art theme over animal theme if available (Information Gathering)
Item 3 Exchange: Camera for a slightly lower resolution, without changing the other options (Information Gathering)
Payment Method: Use Visa card for payment or refund on both orders (Information Gathering)
Cancellation Request: Cancel the skateboard order (Information Gathering)
Cancellation Contingency: If single item cancellation is not possible, I will cancel the whole order myself on the website (Information Gathering)
</multiturn info>
```

**Global/Context Variables:**


**APIs:**

- retail


# Set Up

## Download relevant files

In [39]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"✅ Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.4 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.4.zip (ID: 1TnAaWGfVrMxWTilyhy46-Aue_bh0XkNk)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.4.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.

Generating FC Schemas
✅ Successfully generated 70 FC Schemas to /content/Schemas


## Install Dependencies and Clone Repositories

In [40]:
!pip install -r /content/APIs/requirements.txt

## Import APIs and initiate DBs

In [41]:
# Using default DB for the Tau Benchmark
import retail

retail.SimulationEngine.db.load_state("/content/DBs/RetailDefaultDB.json")


# Initial Assertion

1. A user named Sofia Li exists with the zip code 78260.
2. The user has an order with ID #W3916020 which contains a Bicycle (item '7758198585') and a Jigsaw Puzzle (item '4068787148').
3. The user has an order with ID #W4689314 which contains a Camera (item '5996159312').
4. The user has a Visa card on file with the ID 'credit_card_3951670'.
5. The user has a pending order that includes a skateboard.
6. The initial status of order #W3916020 is 'delivered'.  
7. The initial status of order #W4689314 is 'delivered'.

In [42]:
import retail
from Scripts.assertions_utils import *

# --- Constants for Assertion ---
FIRST_NAME = "Sofia"
LAST_NAME = "Li"
ZIP_CODE = "78260"
ORDER_ID_1 = "#W3916020"
ORDER_1_ITEM_IDS = ["7758198585", "4068787148"]
ORDER_ID_2 = "#W4689314"
ORDER_2_ITEM_ID = "5996159312"
PAYMENT_METHOD_ID = "credit_card_3951670"
PAYMENT_BRAND = "visa"
PENDING_ITEM_NAME = "Skateboard"

# --- Data Gathering from APIs ---
user_id = None
user_details = None
order_1_details = None
order_2_details = None
pending_skateboard_order_found = False
api_error = None

try:
    # Find user and get their details
    user_id = retail.find_user_id_by_name_zip(first_name=FIRST_NAME, last_name=LAST_NAME, zip_code=ZIP_CODE)
    if user_id:
        user_details = retail.get_user_details(user_id=user_id)

    # Get details for the two specified orders
    order_1_details = retail.get_order_details(order_id=ORDER_ID_1)
    order_2_details = retail.get_order_details(order_id=ORDER_ID_2)

    # Check for a pending order with a skateboard
    if user_details and user_details.get("orders"):
        for order_id in user_details["orders"]:
            order_details = retail.get_order_details(order_id=order_id)
            if compare_strings(order_details.get("status"), "pending"):
                items = order_details.get("items", [])
                if any(compare_strings(item.get("name"), PENDING_ITEM_NAME) for item in items):
                    pending_skateboard_order_found = True
                    break
except Exception as e:
    api_error = str(e)

# --- Assertion 1: A user named Sofia Li exists with the zip code 78260. ---
assertion_message_1 = f"Assertion 1 Failed: A user named '{FIRST_NAME} {LAST_NAME}' with zip code '{ZIP_CODE}' was not found."
if api_error:
    assertion_message_1 += f" API Error: {api_error}"
elif user_details:
    actual_name = f"{user_details.get('name', {}).get('first_name')} {user_details.get('name', {}).get('last_name')}"
    actual_zip = user_details.get('address', {}).get('zip')
    assertion_message_1 = (
        f"Assertion 1 Failed: Expected user '{FIRST_NAME} {LAST_NAME}' with zip '{ZIP_CODE}', "
        f"but found '{actual_name}' with zip '{actual_zip}'."
    )

assertion_condition_1 = (
    user_details is not None and
    compare_strings(user_details.get("name", {}).get("first_name"), FIRST_NAME) and
    compare_strings(user_details.get("name", {}).get("last_name"), LAST_NAME) and
    compare_strings(user_details.get("address", {}).get("zip"), ZIP_CODE)
)
assert assertion_condition_1, assertion_message_1

# --- Assertion 2: Order #W3916020 contains expected items. ---
order_1_found_item_ids = []
if order_1_details and order_1_details.get("items"):
    order_1_found_item_ids = [item.get("item_id") for item in order_1_details["items"]]

assertion_message_2 = f"Assertion 2 Failed: Order '{ORDER_ID_1}' did not contain the expected items {ORDER_1_ITEM_IDS}."
if api_error:
    assertion_message_2 += f" API Error: {api_error}"
elif not order_1_details:
    assertion_message_2 = f"Assertion 2 Failed: Order '{ORDER_ID_1}' not found."
else:
    assertion_message_2 = (
        f"Assertion 2 Failed: Order '{ORDER_ID_1}' should contain item IDs {ORDER_1_ITEM_IDS}, "
        f"but found {order_1_found_item_ids}."
    )

assertion_condition_2 = compare_is_list_subset(
    search_value=ORDER_1_ITEM_IDS,
    input_list=order_1_found_item_ids,
    list_comparison_function="all"
)
assert assertion_condition_2, assertion_message_2

# --- Assertion 2b (NEW): Order #W3916020 initial status is 'delivered'. ---
order_1_status = order_1_details.get("status") if order_1_details else None
assertion_message_2b = (
    f"Assertion 2b Failed: Order '{ORDER_ID_1}' initial status is not 'delivered'. "
    f"Actual status: '{order_1_status}'."
)
assertion_condition_2b = compare_strings(order_1_status, "delivered")
assert assertion_condition_2b, assertion_message_2b

# --- Assertion 3: Order #W4689314 contains expected item. ---
order_2_found_item_ids = []
if order_2_details and order_2_details.get("items"):
    order_2_found_item_ids = [item.get("item_id") for item in order_2_details["items"]]

assertion_message_3 = f"Assertion 3 Failed: Order '{ORDER_ID_2}' did not contain the expected item '{ORDER_2_ITEM_ID}'."
if api_error:
    assertion_message_3 += f" API Error: {api_error}"
elif not order_2_details:
    assertion_message_3 = f"Assertion 3 Failed: Order '{ORDER_ID_2}' not found."
else:
    assertion_message_3 = (
        f"Assertion 3 Failed: Order '{ORDER_ID_2}' should contain item ID '{ORDER_2_ITEM_ID}', "
        f"but found {order_2_found_item_ids}."
    )

assertion_condition_3 = compare_is_list_subset(
    search_value=ORDER_2_ITEM_ID,
    input_list=order_2_found_item_ids
)
assert assertion_condition_3, assertion_message_3

# --- Assertion 3b (NEW): Order #W4689314 initial status is 'delivered'. ---
order_2_status = order_2_details.get("status") if order_2_details else None
assertion_message_3b = (
    f"Assertion 3b Failed: Order '{ORDER_ID_2}' initial status is not 'delivered'. "
    f"Actual status: '{order_2_status}'."
)
assertion_condition_3b = compare_strings(order_2_status, "delivered")
assert assertion_condition_3b, assertion_message_3b

# --- Assertion 4: The user has a Visa card on file with the ID 'credit_card_3951670'. ---
payment_method = None
if user_details and user_details.get("payment_methods"):
    payment_method = user_details["payment_methods"].get(PAYMENT_METHOD_ID)

assertion_message_4 = (
    f"Assertion 4 Failed: Payment method '{PAYMENT_METHOD_ID}' not found or is not a '{PAYMENT_BRAND}' card."
)
if api_error:
    assertion_message_4 += f" API Error: {api_error}"
elif payment_method:
    actual_brand = payment_method.get('brand', 'N/A')
    assertion_message_4 = (
        f"Assertion 4 Failed: Expected payment method '{PAYMENT_METHOD_ID}' to be a '{PAYMENT_BRAND}' card, "
        f"but it is a '{actual_brand}' card."
    )

assertion_condition_4 = payment_method is not None and compare_strings(payment_method.get("brand"), PAYMENT_BRAND)
assert assertion_condition_4, assertion_message_4

# --- Assertion 5: The user has a pending order that includes a skateboard. ---
assertion_message_5 = (
    f"Assertion 5 Failed: No pending order found for user '{user_id}' containing an item named '{PENDING_ITEM_NAME}'."
)
if api_error:
    assertion_message_5 += f" API Error: {api_error}"

assertion_condition_5 = pending_skateboard_order_found
assert assertion_condition_5, assertion_message_5


# Action

**Simulated User**: Hi, I need to make a few exchanges. I want to exchange a bicycle for a larger frame size and a jigsaw puzzle for one with 1000 more pieces, preferably with an art theme. I also need to exchange a camera for one with a lower resolution and cancel my skateboard order.

**Action Agent**: I can help with that. What is your email address?

**Simulated User**: I don't have my email handy, but I can give you my name and zip code. It's Sofia Li, and the zip is 78260.

In [43]:
import retail
print(retail.find_user_id_by_name_zip(first_name='Sofia', last_name='Li', zip_code='78260'))

sofia_li_9219


In [44]:
print(retail.get_user_details(user_id='sofia_li_9219'))

{'name': {'first_name': 'Sofia', 'last_name': 'Li'}, 'address': {'address1': '786 Elm Street', 'address2': 'Suite 546', 'city': 'San Antonio', 'country': 'USA', 'state': 'TX', 'zip': '78260'}, 'email': 'sofia.li7352@example.com', 'payment_methods': {'paypal_8194385': {'id': 'paypal_8194385', 'source': 'paypal'}, 'credit_card_3951670': {'id': 'credit_card_3951670', 'source': 'credit_card', 'brand': 'visa', 'last_four': '6791'}, 'credit_card_8105988': {'id': 'credit_card_8105988', 'source': 'credit_card', 'brand': 'mastercard', 'last_four': '8484'}}, 'orders': ['#W4689314', '#W8855135', '#W3916020', '#W5416052']}


In [45]:
print(retail.get_order_details(order_id='#W4689314'))

{'order_id': '#W4689314', 'user_id': 'sofia_li_9219', 'address': {'address1': '786 Elm Street', 'address2': 'Suite 546', 'city': 'San Antonio', 'country': 'USA', 'state': 'TX', 'zip': '78260'}, 'items': [{'name': 'Digital Camera', 'product_id': '8940227892', 'item_id': '5996159312', 'price': 2895.55, 'options': {'resolution': '24MP', 'zoom': '3x', 'storage': 'SD card'}}], 'fulfillments': [{'tracking_id': ['689944590938'], 'item_ids': ['5996159312']}], 'status': 'delivered', 'payment_history': [{'transaction_type': 'payment', 'amount': 2895.55, 'payment_method_id': 'credit_card_8105988'}]}


In [46]:
print(retail.get_order_details(order_id='#W8855135'))

{'order_id': '#W8855135', 'user_id': 'sofia_li_9219', 'address': {'address1': '786 Elm Street', 'address2': 'Suite 546', 'city': 'San Antonio', 'country': 'USA', 'state': 'TX', 'zip': '78260'}, 'items': [{'name': 'Air Purifier', 'product_id': '3821016478', 'item_id': '4035304400', 'price': 504.19, 'options': {'room size': 'medium', 'filter type': 'HEPA', 'features': 'smart sensors'}}, {'name': 'Hiking Boots', 'product_id': '7363354090', 'item_id': '1437889264', 'price': 258.09, 'options': {'size': '7', 'material': 'synthetic', 'waterproof': 'no'}}, {'name': 'Skateboard', 'product_id': '1968349452', 'item_id': '3098764622', 'price': 202.13, 'options': {'deck material': 'plastic', 'length': '34 inch', 'design': 'plain'}}, {'name': 'Yoga Mat', 'product_id': '4635925001', 'item_id': '2733768059', 'price': 94.38, 'options': {'thickness': '6mm', 'material': 'natural rubber', 'color': 'pink'}}], 'fulfillments': [], 'status': 'pending', 'payment_history': [{'transaction_type': 'payment', 'amou

In [47]:
print(retail.get_order_details(order_id='#W3916020'))

{'order_id': '#W3916020', 'user_id': 'sofia_li_9219', 'address': {'address1': '285 Elm Street', 'address2': 'Suite 121', 'city': 'Fort Worth', 'country': 'USA', 'state': 'TX', 'zip': '76155'}, 'items': [{'name': 'Jigsaw Puzzle', 'product_id': '1808611083', 'item_id': '4068787148', 'price': 52.01, 'options': {'pieces': '500', 'theme': 'art', 'difficulty level': 'intermediate'}}, {'name': 'Bicycle', 'product_id': '9783735446', 'item_id': '7758198585', 'price': 1917.21, 'options': {'frame size': 'medium', 'color': 'green', 'type': 'road'}}], 'fulfillments': [{'tracking_id': ['580826001577'], 'item_ids': ['4068787148', '7758198585']}], 'status': 'delivered', 'payment_history': [{'transaction_type': 'payment', 'amount': 1969.22, 'payment_method_id': 'credit_card_8105988'}]}


In [48]:
print(retail.get_product_details(product_id='9783735446'))

{'name': 'Bicycle', 'product_id': '9783735446', 'variants': {'7758198585': {'item_id': '7758198585', 'options': {'frame size': 'medium', 'color': 'green', 'type': 'road'}, 'available': True, 'price': 1917.21}, '5606522780': {'item_id': '5606522780', 'options': {'frame size': 'large', 'color': 'red', 'type': 'mountain'}, 'available': True, 'price': 1902.67}, '6170152315': {'item_id': '6170152315', 'options': {'frame size': 'small', 'color': 'red', 'type': 'mountain'}, 'available': False, 'price': 1814.72}, '3624655057': {'item_id': '3624655057', 'options': {'frame size': 'medium', 'color': 'blue', 'type': 'road'}, 'available': True, 'price': 2195.04}, '2143041831': {'item_id': '2143041831', 'options': {'frame size': 'medium', 'color': 'black', 'type': 'mountain'}, 'available': True, 'price': 2076.5}}}


In [49]:
print(retail.get_product_details(product_id='1808611083'))

{'name': 'Jigsaw Puzzle', 'product_id': '1808611083', 'variants': {'3614853563': {'item_id': '3614853563', 'options': {'pieces': '2000', 'theme': 'art', 'difficulty level': 'intermediate'}, 'available': False, 'price': 46.99}, '4772738468': {'item_id': '4772738468', 'options': {'pieces': '1000', 'theme': 'animals', 'difficulty level': 'beginner'}, 'available': False, 'price': 53.91}, '4068787148': {'item_id': '4068787148', 'options': {'pieces': '500', 'theme': 'art', 'difficulty level': 'intermediate'}, 'available': True, 'price': 52.01}, '3112842858': {'item_id': '3112842858', 'options': {'pieces': '1000', 'theme': 'fantasy', 'difficulty level': 'intermediate'}, 'available': True, 'price': 49.1}, '7869640094': {'item_id': '7869640094', 'options': {'pieces': '2000', 'theme': 'animals', 'difficulty level': 'expert'}, 'available': False, 'price': 47.59}, '1096508426': {'item_id': '1096508426', 'options': {'pieces': '500', 'theme': 'art', 'difficulty level': 'beginner'}, 'available': True

In [50]:
print(retail.get_product_details(product_id='8940227892'))

{'name': 'Digital Camera', 'product_id': '8940227892', 'variants': {'6384525445': {'item_id': '6384525445', 'options': {'resolution': '30MP', 'zoom': '5x', 'storage': 'CF card'}, 'available': True, 'price': 2929.62}, '3892645120': {'item_id': '3892645120', 'options': {'resolution': '30MP', 'zoom': '10x', 'storage': 'CF card'}, 'available': False, 'price': 3070.64}, '1804581713': {'item_id': '1804581713', 'options': {'resolution': '30MP', 'zoom': '3x', 'storage': 'SD card'}, 'available': True, 'price': 2875.61}, '9644439410': {'item_id': '9644439410', 'options': {'resolution': '20MP', 'zoom': '5x', 'storage': 'CF card'}, 'available': True, 'price': 3280.31}, '5996159312': {'item_id': '5996159312', 'options': {'resolution': '24MP', 'zoom': '3x', 'storage': 'SD card'}, 'available': True, 'price': 2895.55}, '7255224608': {'item_id': '7255224608', 'options': {'resolution': '30MP', 'zoom': '3x', 'storage': 'CF card'}, 'available': True, 'price': 2922.97}, '2284404181': {'item_id': '228440418

**Action Agent**: Thanks, Sofia. I’ve found your account and reviewed your orders:

* Order #W3916020: Bicycle and Jigsaw Puzzle.
* Order #W4689314: Digital Camera.
* Order #W8855135: Skateboard and other items.

Which payment method should I use for any refunds or additional charges?

**Simulated User**: Please use my Visa ending in 6791 for all transactions.

**Action Agent**: Perfect! Here’s what I’ll do:

1. Exchange your **Bicycle** for a **large frame, red, mountain bike**.
2. Exchange your **Jigsaw Puzzle** for a **1500-piece art-themed puzzle**.
3. Exchange your **Camera** for a **20MP version**.
4. Regarding the **Skateboard**, our system doesn’t support canceling individual items. I have to cancel the entire order.

The refunds and any adjustments will be processed to your Visa card ending in 6791. Do you want me to proceed?

**Simulated User**: Do not cancel the skateboard order. I'll do that myself. Exchange the rest of the items.

**Action Agent**: Great! Processing the exchanges now.

In [51]:
print(retail.exchange_delivered_order_items(order_id='#W3916020', item_ids=['7758198585', '4068787148'], new_item_ids=['5606522780', '5546244844'], payment_method_id='credit_card_3951670'))

{'order_id': '#W3916020', 'user_id': 'sofia_li_9219', 'address': {'address1': '285 Elm Street', 'address2': 'Suite 121', 'city': 'Fort Worth', 'country': 'USA', 'state': 'TX', 'zip': '76155'}, 'items': [{'name': 'Jigsaw Puzzle', 'product_id': '1808611083', 'item_id': '4068787148', 'price': 52.01, 'options': {'pieces': '500', 'theme': 'art', 'difficulty level': 'intermediate'}}, {'name': 'Bicycle', 'product_id': '9783735446', 'item_id': '7758198585', 'price': 1917.21, 'options': {'frame size': 'medium', 'color': 'green', 'type': 'road'}}], 'fulfillments': [{'tracking_id': ['580826001577'], 'item_ids': ['4068787148', '7758198585']}], 'status': 'exchange requested', 'payment_history': [{'transaction_type': 'payment', 'amount': 1969.22, 'payment_method_id': 'credit_card_8105988'}], 'exchange_items': ['4068787148', '7758198585'], 'exchange_new_items': ['5546244844', '5606522780'], 'exchange_payment_method_id': 'credit_card_3951670', 'exchange_price_difference': -14.96}


In [52]:
print(retail.exchange_delivered_order_items(order_id='#W4689314', item_ids=['5996159312'], new_item_ids=['8363011723'], payment_method_id='credit_card_3951670'))

{'order_id': '#W4689314', 'user_id': 'sofia_li_9219', 'address': {'address1': '786 Elm Street', 'address2': 'Suite 546', 'city': 'San Antonio', 'country': 'USA', 'state': 'TX', 'zip': '78260'}, 'items': [{'name': 'Digital Camera', 'product_id': '8940227892', 'item_id': '5996159312', 'price': 2895.55, 'options': {'resolution': '24MP', 'zoom': '3x', 'storage': 'SD card'}}], 'fulfillments': [{'tracking_id': ['689944590938'], 'item_ids': ['5996159312']}], 'status': 'exchange requested', 'payment_history': [{'transaction_type': 'payment', 'amount': 2895.55, 'payment_method_id': 'credit_card_8105988'}], 'exchange_items': ['5996159312'], 'exchange_new_items': ['8363011723'], 'exchange_payment_method_id': 'credit_card_3951670', 'exchange_price_difference': -71.59}


**Action Agent**: I have successfully exchanged your items as requested.

Do you want to make a new order?

# Final Assertion

1. The status of order #W3916020 is `exchange requested`.
2. The status of order #W4689314 is `exchange requested`.
4. The order containing the skateboard has its status 'pending'.
4. Order #W3916020 exchanged items correctly — old item IDs are removed and new item IDs are present.  
5. Order #W4689314 exchanged items correctly — old item IDs are removed and new item IDs are present.


In [57]:
import retail
from Scripts.assertions_utils import *

# --- Constants ---
ORDER_ID_1 = '#W3916020'
EXCHANGED_ITEMS_1 = ['7758198585', '4068787148']
NEW_ITEMS_1 = ['5606522780', '5546244844']

ORDER_ID_2 = '#W4689314'
EXCHANGED_ITEMS_2 = ['5996159312']
NEW_ITEMS_2 = ['8363011723']

USER_FIRST_NAME = 'Sofia'
USER_LAST_NAME = 'Li'
USER_ZIP_CODE = '78260'
CANCELLED_ITEM_NAME = 'Skateboard'

# --- Data Gathering ---
order_1_details = None
order_2_details = None
skateboard_order_details = None
api_error_log = []

try:
    order_1_details = retail.get_order_details(order_id=ORDER_ID_1)
except Exception as e:
    api_error_log.append(f"Error fetching details for order {ORDER_ID_1}: {e}")

try:
    order_2_details = retail.get_order_details(order_id=ORDER_ID_2)
except Exception as e:
    api_error_log.append(f"Error fetching details for order {ORDER_ID_2}: {e}")

try:
    user_id = retail.find_user_id_by_name_zip(
        first_name=USER_FIRST_NAME,
        last_name=USER_LAST_NAME,
        zip_code=USER_ZIP_CODE
    )
    if user_id:
        user_details = retail.get_user_details(user_id=user_id)
        user_orders = user_details.get('orders', [])
        for order_id in user_orders:
            if order_id in [ORDER_ID_1, ORDER_ID_2]:
                continue
            try:
                current_order_details = retail.get_order_details(order_id=order_id)
                items = current_order_details.get('items', [])
                if any(compare_is_string_subset(CANCELLED_ITEM_NAME, item.get('name', '')) for item in items):
                    skateboard_order_details = current_order_details
                    break
            except Exception as e:
                api_error_log.append(f"Error fetching details for potential skateboard order {order_id}: {e}")
except Exception as e:
    api_error_log.append(f"Error identifying user {USER_FIRST_NAME} {USER_LAST_NAME}: {e}")

# --- Assertion 1: Order #W3916020 has status 'exchange requested' ---
assertion_condition_1 = order_1_details and compare_strings(order_1_details.get('status'), 'exchange requested')
assertion_message_1 = (
    f"Assertion 1 Failed: Order {ORDER_ID_1} status is not 'exchange requested'. "
    f"Actual status: '{order_1_details.get('status') if order_1_details else 'None'}'."
)
assert assertion_condition_1, assertion_message_1

# --- Assertion 2: Order #W4689314 has status 'exchange requested' ---
assertion_condition_2 = order_2_details and compare_strings(order_2_details.get('status'), 'exchange requested')
assertion_message_2 = (
    f"Assertion 2 Failed: Order {ORDER_ID_2} status is not 'exchange requested'. "
    f"Actual status: '{order_2_details.get('status') if order_2_details else 'None'}'."
)
assert assertion_condition_2, assertion_message_2

# --- Assertion 3: Skateboard order status remains 'pending' ---
skateboard_order_status = skateboard_order_details.get('status') if skateboard_order_details else None
assertion_condition_3 = compare_strings(skateboard_order_status, 'pending')
assertion_message_3 = (
    f"Assertion 3 Failed: The skateboard order status is not 'pending'. "
)
if not skateboard_order_details:
    assertion_message_3 += (
        f"No order containing '{CANCELLED_ITEM_NAME}' was found for user {USER_FIRST_NAME} {USER_LAST_NAME}."
    )
    if api_error_log:
        assertion_message_3 += f" API Errors: {api_error_log}"
else:
    assertion_message_3 += f"Found status: '{skateboard_order_status}'."
assert assertion_condition_3, assertion_message_3

# --- Assertion 4: #W3916020 exchanged items correctly ---
exchange_old_1 = [str(x) for x in order_1_details.get('exchange_items', [])]
exchange_new_1 = [str(x) for x in order_1_details.get('exchange_new_items', [])]

assertion_condition_4_old = compare_is_list_subset(EXCHANGED_ITEMS_1, exchange_old_1, list_comparison_function="all")
assertion_condition_4_new = compare_is_list_subset(NEW_ITEMS_1, exchange_new_1, list_comparison_function="all")
assertion_message_4 = (
    f"Assertion 4 Failed: Order {ORDER_ID_1} exchange items mismatch. "
    f"Expected old {EXCHANGED_ITEMS_1} in {exchange_old_1} and new {NEW_ITEMS_1} in {exchange_new_1}."
)
assert assertion_condition_4_old and assertion_condition_4_new, assertion_message_4

# --- Assertion 5: #W4689314 exchanged items correctly ---
exchange_old_2 = [str(x) for x in order_2_details.get('exchange_items', [])]
exchange_new_2 = [str(x) for x in order_2_details.get('exchange_new_items', [])]

assertion_condition_5_old = compare_is_list_subset(EXCHANGED_ITEMS_2, exchange_old_2, list_comparison_function="all")
assertion_condition_5_new = compare_is_list_subset(NEW_ITEMS_2, exchange_new_2, list_comparison_function="all")
assertion_message_5 = (
    f"Assertion 5 Failed: Order {ORDER_ID_2} exchange items mismatch. "
    f"Expected old {EXCHANGED_ITEMS_2} in {exchange_old_2} and new {NEW_ITEMS_2} in {exchange_new_2}."
)
assert assertion_condition_5_old and assertion_condition_5_new, assertion_message_5
